# SDM on the ResNet-18, with 100 calibration images per class

This repeats [notebook 03](03_sdm_resnet18.ipynb) on the ResNet from [notebook 04](04_cifar100_resnet18_cal100.ipynb), which has **100 calibration images per class** instead of 25. In notebook 03, SDM formed no region on the 100 classes, and too few calibration images was the main suspect.

**Goal:** apply the **Similarity-Distance-Magnitude (SDM)** estimator to the ResNet-18 trained in [notebook 04](04_cifar100_resnet18_cal100.ipynb), and compare it with the temperature-scaling baseline on the same images.

SDM decides whether to trust a prediction by looking at the model's **embedding** of the image:

- **Similarity**: do the nearest training images (in embedding space) share the predicted label?
- **Distance**: how far is the image from the training images it resembles?
- **Magnitude**: how strongly does the output layer favor the predicted class?

It then calibrates these signals on the calibration split and assigns each image an **α**: an estimate that the image falls in a region where accuracy *for its class and predicted class* is at least α. An image with α = 0 belongs to no region and should be treated as out-of-distribution. We use the [`reexpress-sdm`](https://pypi.org/project/reexpress-sdm/) package, version 0.4.9.

This notebook:

1. rebuilds the trained ResNet-18 from `outputs/<MODEL_NAME>.pt` and computes (or loads from cache) embeddings for the training, calibration, and validation splits,
2. **Experiment A**: SDM on all **100 classes**, with 100 calibration images per class,
3. **Experiment B**: SDM on the **20 broader groups**, a pilot with five times more calibration images per label,
4. shows the nearest training images SDM uses to justify its decisions.

**Before running:** run notebook 04 first, so the checkpoint exists. Install the extra package with `python -m pip install -r requirements.txt`. The first run takes roughly 15 minutes on an Apple M3 Pro, mostly two SDM fits. Embeddings and fitted SDM models are cached in `outputs/`, so later runs take about a minute.

**Evaluation audit — October 29, 2026:** thresholds and risk curves now use the shared whole-confidence-group metrics. Saved outputs are cleared because historical tables used older definitions. Calibration-matched comparisons freeze the coverage budget on calibration only; validation coverages may differ. This source repair does not rerun or validate historical benchmark claims.


## 1. Settings and tools

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/sdm_vision").is_dir())
sys.path.insert(0, str(REPO_ROOT / "src"))
from sdm_vision.metrics import choose_threshold
from sdm_vision.sdm import validate_calibration_identity
import hashlib
import pickle
import tarfile
import time

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import reexpress_sdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from reexpress_sdm import SDMModel, TorchTrainer, TrainingConfig, load_artifact, write_artifact

SDM_EPOCHS = 20        # the package default
SDM_SEED = 0
TARGET_ALPHA = 0.95    # the reliability level we want, as in notebook 02
MODEL_NAME = "cifar100_resnet18_cal100"  # saved by notebook 04; notebook 03 uses "cifar100_resnet18" (25 calibration images per class)
SAVE_SDM_MODELS = True    # cache fitted SDM models in outputs/ (about 190 MB each) and reload them instead of refitting (about 5 minutes each)
REBUILD_EMBEDDINGS = False  # True recomputes the embeddings even when a matching cache exists
OUTPUT_DIR = Path("../outputs")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"PyTorch {torch.__version__}, reexpress-sdm {reexpress_sdm.__version__}, device: {device}")

## 2. Load the data and the trained ResNet-18

The data loader is the same as in the earlier notebooks. The network class must match notebook 04 exactly so the saved weights fit. The checkpoint also stores the split indices, so we use exactly the same training, calibration, and validation images.

In [ ]:
DATA_DIR = Path("../data")  # the repository's data/ folder
data_path = DATA_DIR / "cifar-100-python"

if not data_path.exists():
    archive_path = DATA_DIR / "cifar-100-python.tar.gz"
    if archive_path.exists():
        with tarfile.open(archive_path) as archive:
            archive.extractall(DATA_DIR, filter="data")
    else:
        data_path = Path(kagglehub.dataset_download("aakash119/cifar-100-python-tar-gz")) / "cifar-100-python"

expected_md5 = {
    "train": "16019d7e3df5f24257cddd939b257f8d",
    "meta": "7973b15100ade9c7d40fb424638fde48",
}
for name, md5 in expected_md5.items():
    assert hashlib.md5((data_path / name).read_bytes()).hexdigest() == md5, f"{name} is not the official file; delete {data_path} and rerun"

with (data_path / "train").open("rb") as file:
    training = pickle.load(file, encoding="latin1")
with (data_path / "meta").open("rb") as file:
    metadata = pickle.load(file, encoding="latin1")
all_images = training["data"].reshape(-1, 3, 32, 32)
all_labels = np.array(training["fine_labels"])
all_groups = np.array(training["coarse_labels"])
class_names = metadata["fine_label_names"]
group_names = metadata["coarse_label_names"]

In [ ]:
# Same architecture as notebook 04.
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class CifarResNet(nn.Module):
    def __init__(self, num_classes=100, widths=(64, 128, 256, 512)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, padding=1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        stages, in_channels = [], widths[0]
        for i, width in enumerate(widths):
            stride = 1 if i == 0 else 2
            stages.append(nn.Sequential(ResidualBlock(in_channels, width, stride), ResidualBlock(width, width)))
            in_channels = width
        self.stages = nn.Sequential(*stages)
        self.classifier = nn.Linear(widths[-1], num_classes)

    def embed(self, x):
        return F.adaptive_avg_pool2d(self.stages(self.stem(x)), 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.embed(x))


checkpoint_path = OUTPUT_DIR / f"{MODEL_NAME}.pt"
checkpoint_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()  # identifies this exact ResNet for the caches
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = CifarResNet(num_classes=len(class_names)).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()
channel_mean = checkpoint["channel_mean"].to(device)
channel_std = checkpoint["channel_std"].to(device)
splits = checkpoint["split_indices"]          # train / validation / calibration indices into the training file
TEMPERATURE = checkpoint["temperature"]       # fitted in notebook 04 on the calibration split
print({name: len(index) for name, index in splits.items()}, f"temperature {TEMPERATURE:.2f}")

## 3. Compute embeddings

SDM works on the **512-number embedding** each image gets just before the ResNet's final layer. We compute it, plus the logits, for every image in each split. No augmentation is applied.

The result is cached in `outputs/<MODEL_NAME>_embeddings.npz` together with a fingerprint (SHA-256) of the checkpoint. A later run loads the cache instead of recomputing, but only if the fingerprint and the split indices match, so a retrained ResNet never reuses stale embeddings. Set `REBUILD_EMBEDDINGS = True` to force a recompute.

The **training** embeddings become SDM's *support set*: the reference images it searches for nearest neighbors. The **calibration** embeddings fit SDM's regions. The **validation** embeddings are only used to check the result.

In [ ]:
@torch.no_grad()
def embed_images(images, batch_size=500):
    embeddings, logits = [], []
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(images[start : start + batch_size]).to(device)
        features = model.embed((batch.float() / 255 - channel_mean) / channel_std)
        embeddings.append(features.cpu())
        logits.append(model.classifier(features).cpu())
    return torch.cat(embeddings).numpy(), torch.cat(logits)


embeddings_path = OUTPUT_DIR / f"{MODEL_NAME}_embeddings.npz"


def load_cached_embeddings():
    # The cache is valid only for this exact checkpoint and these exact splits.
    if REBUILD_EMBEDDINGS or not embeddings_path.exists():
        return None
    with np.load(embeddings_path) as cache:
        if "checkpoint_sha256" not in cache.files or str(cache["checkpoint_sha256"]) != checkpoint_sha256:
            return None
        if any(not np.array_equal(cache[f"{name}_index"], index) for name, index in splits.items()):
            return None
        return {name: {"index": cache[f"{name}_index"], "embeddings": cache[f"{name}_embeddings"],
                       "logits": torch.from_numpy(cache[f"{name}_logits"]),
                       "labels": cache[f"{name}_labels"], "groups": cache[f"{name}_groups"]}
                for name in splits}


data = load_cached_embeddings()
if data is None:
    data = {}
    for name, index in splits.items():
        embeddings, logits = embed_images(all_images[index])
        data[name] = {"index": index, "embeddings": embeddings, "logits": logits,
                      "labels": all_labels[index], "groups": all_groups[index]}
    np.savez_compressed(
        embeddings_path, checkpoint_sha256=checkpoint_sha256,
        **{f"{name}_{key}": (value.numpy() if torch.is_tensor(value) else value)
           for name, split in data.items() for key, value in split.items()},
    )
    print("Computed embeddings and cached them in", embeddings_path)
else:
    print("Loaded cached embeddings from", embeddings_path)

pd.DataFrame({
    name: {"images": len(split["labels"]),
           "ResNet accuracy": f"{(split['logits'].argmax(1).numpy() == split['labels']).mean():.1%}"}
    for name, split in data.items()
}).T

The ResNet is right on about 99% of its **training** images, because it has nearly memorized them, but only about 74% on validation. Keep this in mind: SDM compares new images with these training embeddings, which are much more tightly clustered than embeddings of new images.

## 4. Tools shared by both experiments

- `fit_sdm` trains SDM on the training embeddings and calibrates it on the calibration embeddings. Calling the trainer directly keeps **our** splits; the command-line tool would otherwise pool and reshuffle them. With `SAVE_SDM_MODELS = True` each fitted model is cached in `outputs/` (about 190 MB, mostly a copy of the support embeddings) and reloaded on the next run. The cache is used only if the model's representation fingerprint matches the current checkpoint.
- `SDMModel(artifact)` builds SDM's nearest-neighbor index over the training images once. We keep one per experiment in memory (`fine_scorer`, `coarse_scorer`) and reuse it for every scoring call.
- `region_blockers` explains a model with no regions (see Experiment A).
- `selective_summary` reports coverage and selective accuracy with a 95% Wilson interval, as in notebook 02, plus the **worst per-label accuracy** among admitted images, because SDM's calibration criterion is class-conditional.

In [ ]:
def fit_sdm(number_of_classes, train_labels, calibration_labels, label_names, name):
    path = OUTPUT_DIR / f"{MODEL_NAME}_{name}_epochs{SDM_EPOCHS}_seed{SDM_SEED}.sdmkitmodel"
    fingerprint = f"{MODEL_NAME}_avgpool512_{checkpoint_sha256[:16]}"
    start = time.time()
    artifact = load_artifact(path) if SAVE_SDM_MODELS and path.exists() else None
    if artifact is not None and artifact.manifest["representation"]["fingerprint"] == fingerprint:
        validate_calibration_identity(artifact, [f"train-{i}" for i in data["calibration"]["index"]], calibration_labels, label_names)
        source = f"loaded from {path.name}"
    else:
        trainer = TorchTrainer(TrainingConfig(number_of_classes=number_of_classes, epochs=SDM_EPOCHS, seed=SDM_SEED), device=str(device))
        result = trainer.fit(
            data["train"]["embeddings"], train_labels.tolist(),
            data["calibration"]["embeddings"], calibration_labels.tolist(),
            train_ids=[f"train-{i}" for i in data["train"]["index"]],
            calibration_ids=[f"train-{i}" for i in data["calibration"]["index"]],
            class_names=label_names,
            representation_fingerprint=fingerprint,
        )
        artifact = result.artifact
        source = f"best epoch {result.best_epoch}"
        if SAVE_SDM_MODELS:
            write_artifact(path, artifact, overwrite=path.exists())
    regions = [round(region["alpha"], 2) for region in artifact.manifest["regions"]]
    print(f"{name}: {source}, {time.time() - start:.0f}s, regions at alpha = {regions or 'none'}")
    return artifact


def region_ladder(artifact, number_of_classes):
    # Replays SDM's nested calibration. At each alpha of the ladder, find the similarity cutoff that leaves
    # the fewest failing labels among the calibration images no earlier region took; if SDM formed a region
    # at that alpha, remove the images it accepts before moving on, exactly as the package does.
    rows = artifact.calibration_rows
    probs = np.array([row["sdm"] for row in rows], dtype=np.float32)
    q_prime = np.array([row["qPrime"] for row in rows], dtype=np.float32)
    labels = np.array([int(row["label"]) for row in rows])
    predictions = np.array([int(row["prediction"]) for row in rows])
    p_true = probs[np.arange(len(labels)), labels]
    configuration = artifact.manifest["configuration"]
    regions = {round(region["alpha"], 10): region for region in artifact.manifest["regions"]}
    residual = np.floor(q_prime) > configuration["oodLimit"]
    ladder = {}
    for step in range(1, round(0.5 / configuration["alphaResolution"]) + 1):
        alpha = round(1 - step * configuration["alphaResolution"], 10)
        if alpha <= 0.5 or not residual.any():
            break
        below = p_true < np.float32(alpha)
        best = None
        for cutoff in np.unique(q_prime[residual]):
            kept = residual & (q_prime >= cutoff)
            count = np.bincount(labels[kept], minlength=number_of_classes)
            misses = np.bincount(labels[kept & below], minlength=number_of_classes)
            allowed = np.floor(round(1 - alpha, 10) * count).astype(int)
            failing = (count == 0) | (misses > allowed)
            if best is None or failing.sum() < best["failing"].sum():
                best = {"failing": failing, "count": count, "misses": misses, "allowed": allowed}
        region = regions.get(alpha)
        ladder[alpha] = best | {"region": region, "rows left": int(residual.sum())}
        if region is not None:
            above = probs >= np.asarray(region["outputThresholds"], dtype=np.float32)
            accepted = (residual & (q_prime >= np.float32(region["minimumRescaledSimilarity"]))
                        & (above.sum(1) == 1) & above[np.arange(len(labels)), predictions])
            residual &= ~accepted
    return ladder


def region_blockers(artifact, number_of_classes):
    # For each alpha, the fewest failing labels at the best cutoff; 0 failing means SDM formed a region there.
    if not any(np.floor(row["qPrime"]) > artifact.manifest["configuration"]["oodLimit"] for row in artifact.calibration_rows):
        return pd.DataFrame([{"region formed": "", "calibration images left": 0,
                              "labels with no images left": number_of_classes, "labels with too many misses": 0}],
                            index=pd.Index(["none eligible"], name="alpha"))
    table = {}
    for alpha, step in region_ladder(artifact, number_of_classes).items():
        table[f"{alpha:.2f}"] = {
            "region formed": "yes" if step["region"] else "",
            "calibration images left": step["rows left"],
            "labels with no images left": int((step["count"] == 0).sum()),
            "labels with too many misses": int(((step["count"] > 0) & (step["misses"] > step["allowed"])).sum()),
        }
    return pd.DataFrame(table).T.rename_axis("alpha")


def wilson_interval(successes, n, z=1.96):
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    half_width = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator
    return center - half_width, center + half_width


def selective_summary(admitted, predictions, labels):
    n = int(admitted.sum())
    correct = predictions[admitted] == labels[admitted]
    if n == 0:
        return {"coverage": "0.0%", "images answered": 0, "selective accuracy": "-", "95% interval": "-", "worst label accuracy": "-"}
    low, high = wilson_interval(correct.sum(), n)
    per_label = pd.Series(correct).groupby(labels[admitted]).mean()
    return {
        "coverage": f"{admitted.mean():.1%}",
        "images answered": n,
        "selective accuracy": f"{correct.mean():.1%}",
        "95% interval": f"{low:.1%} – {high:.1%}",
        "worst label accuracy": f"{per_label.min():.0%} ({len(per_label)} labels)",
    }




def score_validation(sdm, nearest_exemplars=0):
    # `sdm` is an SDMModel kept in memory, so its nearest-neighbor index is built only once.
    scores = sdm.score(data["validation"]["embeddings"], ids=[f"train-{i}" for i in data["validation"]["index"]],
                       nearest_exemplars=nearest_exemplars)
    return {
        "scores": scores,
        "prediction": np.array([score.prediction for score in scores]),
        "centroid_alpha": np.array([score.centroid_region_alpha for score in scores]),
        "lower_alpha": np.array([score.lower_region_alpha for score in scores]),
        "is_ood": np.array([score.is_ood for score in scores]),
    }

## 5. Experiment A: SDM on all 100 classes

SDM's calibration creates a region at level α only if there is **one similarity cutoff at which every class at once**:

1. still has at least one calibration image above the cutoff, and
2. has nearly all of those images given at least α probability for their true class. At α = 0.95 that allows 5 misses in 100 images, but only 1 in 25 and **no misses** once fewer than 20 images remain.

Notebook 03 had only **25 calibration images per class**, and SDM formed no region at all. This notebook has **100 per class**, so each class can lose more images to a strict cutoff and still tolerate a few misses.

In [ ]:
sdm_fine = fit_sdm(100, data["train"]["labels"], data["calibration"]["labels"], class_names, "sdm_fine100")
fine_scorer = SDMModel(sdm_fine, device=str(device))
fine = score_validation(fine_scorer)
y_fine = data["validation"]["labels"]

print(f"SDM's own prediction accuracy on validation: {(fine['prediction'] == y_fine).mean():.1%} "
      f"(the ResNet itself: {(data['validation']['logits'].argmax(1).numpy() == y_fine).mean():.1%})")
print(f"validation images flagged out-of-distribution by SDM: {fine['is_ood'].mean():.1%}")
print(f"validation images in any region (alpha > 0): {(fine['centroid_alpha'] > 0).mean():.1%}")

If no region was formed, every image gets α = 0 and SDM admits nothing. The table below shows why, or how close each α came: for each α, the best possible cutoff and how many classes still fail.

In [ ]:
region_blockers(sdm_fine, 100)

**Reading the table:** a region needs **0 failing classes** in a row. SDM builds its regions as a ladder from high α to low: once a region forms, the calibration images it accepts are set aside, and the lower α levels are checked on the images that are left (the "calibration images left" column). "No images left" means a cutoff strict enough to drop the misses also removes *every* calibration image of some classes; the fewer images a class has, the sooner that happens. "Too many misses" means even the best cutoff leaves a class with more low-probability images than α allows.

In notebook 03 (25 calibration images per class, no regions), two causes stood out:

- **Too few calibration images per class.** With 25 images, the per-class rule tolerates at most 1 miss. This notebook addresses that with 100 per class.
- **Memorized training embeddings.** SDM's first gate requires the nearest training images to agree with the prediction. Because training images are tightly clustered by class, many new images land next to a different class and fail that gate.

Compare this table with the one in [notebook 03](03_sdm_resnet18.ipynb): if the "no images left" column shrinks but "too many misses" remains, the second cause dominates.

## 6. Experiment B: SDM on the 20 broader groups

We keep the same embeddings and the same images, but use the **20 broader groups** (for example *trees*, *vehicles 1*) as labels. Each group has **500 calibration images** instead of 100. Groups are an easier task than classes, so this shows how SDM behaves when each label has plenty of calibration data.

For a fair comparison, the softmax baseline is converted to groups too: a group's probability is the sum of the temperature-scaled probabilities of its five classes. Its thresholds are chosen on the calibration split exactly as in notebook 02.

In [ ]:
sdm_coarse = fit_sdm(20, data["train"]["groups"], data["calibration"]["groups"], group_names, "sdm_coarse20")
coarse_scorer = SDMModel(sdm_coarse, device=str(device))
coarse = score_validation(coarse_scorer, nearest_exemplars=5)
y_coarse = data["validation"]["groups"]
print(f"SDM's own prediction accuracy on the 20 groups: {(coarse['prediction'] == y_coarse).mean():.1%}")

In [ ]:
# Temperature-scaled softmax, summed into the 20 groups.
fine_to_group = np.zeros(100, dtype=int)
fine_to_group[all_labels] = all_groups

def group_probabilities(logits):
    probs = (logits / TEMPERATURE).softmax(1).numpy()
    grouped = np.zeros((len(probs), 20))
    np.add.at(grouped.T, fine_to_group, probs.T)
    return grouped

cal_group_probs = group_probabilities(data["calibration"]["logits"])
val_group_probs = group_probabilities(data["validation"]["logits"])
cal_correct = cal_group_probs.argmax(1) == data["calibration"]["groups"]
val_confidence, val_prediction = val_group_probs.max(1), val_group_probs.argmax(1)

rows = {}
for rule, conservative in [("softmax + temperature, plain", False), ("softmax + temperature, conservative", True)]:
    threshold = choose_threshold(cal_group_probs.max(1), cal_correct, TARGET_ALPHA, conservative)
    admitted = val_confidence >= threshold if threshold is not None else np.zeros(len(y_coarse), bool)
    rows[rule] = selective_summary(admitted, val_prediction, y_coarse)
for alpha in [0.90, 0.95]:
    rows[f"SDM, centroid alpha ≥ {alpha:.2f}"] = selective_summary(coarse["centroid_alpha"] >= alpha, coarse["prediction"], y_coarse)
    rows[f"SDM, lower alpha ≥ {alpha:.2f}"] = selective_summary(coarse["lower_alpha"] >= alpha, coarse["prediction"], y_coarse)

pd.DataFrame(rows).T.rename_axis("validation, 20 groups")

**How to read this table:**

- **Coverage** is the share of validation images the method answers; **selective accuracy** is the accuracy on those. A better method answers more images at the same accuracy.
- **Worst label accuracy** is the lowest accuracy among the groups among admitted images. SDM aims to keep *every* group at α or above, not just the average.
- The softmax thresholds aim at exactly 95% overall. SDM's regions are deliberately stricter: per label, with a margin. So compare SDM at α ≥ 0.95 with the **conservative** softmax row, and look at both coverage and the worst-label column.

### Which groups does SDM admit?

Coverage per group, for SDM at α ≥ 0.95 and for the conservative softmax rule. Groups the model finds hard should be admitted less.

In [ ]:
conservative_threshold = choose_threshold(cal_group_probs.max(1), cal_correct, TARGET_ALPHA, conservative=True)
per_group = pd.DataFrame({
    "SDM α ≥ 0.95": pd.Series(coarse["centroid_alpha"] >= TARGET_ALPHA).groupby(y_coarse).mean(),
    "softmax, conservative": pd.Series(val_confidence >= (conservative_threshold or np.inf)).groupby(y_coarse).mean(),
    "ResNet group accuracy": pd.Series(val_prediction == y_coarse).groupby(y_coarse).mean(),
})
per_group.index = [group_names[i].replace("_", " ") for i in per_group.index]
ax = per_group.sort_values("ResNet group accuracy").plot.barh(figsize=(9, 7), color=["#287c8e", "#c4553a", "#bbbbbb"])
ax.set(xlabel="fraction of the group's validation images", title="Coverage per group (sorted by accuracy)")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.tight_layout()
plt.show()

## 7. Interpretability by example: the nearest training images

For every decision, SDM can show the **training images closest to the query** in its embedding space. Each row below shows one validation image (left) and its five nearest training images, with their true group. The top rows were admitted at α ≥ 0.95. The bottom rows were rejected (α = 0), so their neighbors often disagree.

In [ ]:
rng = np.random.default_rng(0)
admitted_ids = rng.choice(np.flatnonzero(coarse["centroid_alpha"] >= TARGET_ALPHA), size=min(3, np.count_nonzero(coarse["centroid_alpha"] >= TARGET_ALPHA)), replace=False)
rejected_ids = rng.choice(np.flatnonzero(coarse["centroid_alpha"] == 0), size=min(3, np.count_nonzero(coarse["centroid_alpha"] == 0)), replace=False)
train_index = data["train"]["index"]

fig, axes = plt.subplots(6, 6, figsize=(10, 10.5))
for axis in axes.flat:
    axis.axis("off")
for row, (i, status) in enumerate([(i, "admitted") for i in admitted_ids] + [(i, "rejected") for i in rejected_ids]):
    query = data["validation"]["index"][i]
    axes[row, 0].imshow(all_images[query].transpose(1, 2, 0), interpolation="nearest")
    axes[row, 0].set_title(f"{status}\ntrue: {group_names[y_coarse[i]]}\npred: {group_names[coarse['prediction'][i]]}", fontsize=7)
    for ax, match in zip(axes[row, 1:], coarse["scores"][i].nearest_support_matches):
        neighbor = train_index[match.support_index]
        ax.imshow(all_images[neighbor].transpose(1, 2, 0), interpolation="nearest")
        ax.set_title(group_names[match.label], fontsize=7, color="#287c8e" if match.label == coarse["prediction"][i] else "#c4553a")
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle("Validation image (left) and its 5 nearest training images\n(teal = same group as the prediction, red = different)", fontsize=12)
plt.tight_layout()
plt.show()

## 8. What we learned and what to try next

Write your observations here. Questions to answer:

1. **Experiment A:** with 100 calibration images per class, did SDM form regions on 100 classes (notebook 03, with 25, formed none)? Which cause dominates in the blocker table: classes running out of images, or too many misses?
2. **Experiment B:** at α ≥ 0.95, how does SDM's coverage compare with the conservative softmax rule? Is its worst-group accuracy higher?
3. Do the nearest training images make the rejected cases understandable?

Next experiments this suggests:

- **Even more calibration images per class**, if 100 per class is still not enough.
- **A less memorized embedding**: stop ResNet training earlier, or add regularization, so training embeddings look more like those of new images.
- **Distribution shift**: score CIFAR-100-C and SVHN embeddings with the cached SDM scorers (`fine_scorer`, `coarse_scorer`) and compare admission rates with the softmax rule.